# CatBoost — third model for the blend

Same data pipeline / CV / sample_weight as `xgboost.ipynb` and `lightgbm.ipynb`. Aim is **diversity, not raw single-model AUC** — the existing XGB/LGBM pair correlates at 0.9905 so the blend has tapped out. CatBoost uses a different inductive bias (ordered target stats for categoricals + oblivious trees + ordered boosting) which should produce uncorrelated errors.

**Metric:** ROC-AUC. **Target:** `pitnextlap`.

## Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.figsize'] = (12, 6)

import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

import catboost as cb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

from common import *

print('catboost:', cb.__version__)

catboost: 1.2.10


## Load + preprocess (mirror of xgboost.ipynb / lightgbm.ipynb)

In [2]:
train_df = pd.read_csv('archive/train.csv')
test_df  = pd.read_csv('archive/test.csv')
orig_df  = pd.read_csv('archive/f1_strategy_dataset_v4.csv')

df = (train_df
        .pipe(copy_data)
        .pipe(clean_data)
        .pipe(remove_duplicates)
        .pipe(make_new_features))

orig_df_cleaned = (orig_df
        .pipe(copy_data)
        .pipe(clean_data)
        .pipe(remove_duplicates)
        .pipe(make_new_features))

train_df_length = df.shape[0]
df = pd.concat([df, orig_df_cleaned])
if 'normalized_tyrelife' in df.columns:
    df = df.drop('normalized_tyrelife', axis=1)
df = df.reset_index(drop=True)

sample_weights = np.ones(df.shape[0])
sample_weights[train_df_length:] = 1.25

target = get_target()
features = get_features(df)
categorical_features = ['compound', 'race', 'year_cat']

# CatBoost wants categoricals as strings (it converts internally otherwise; explicit is safer).
# NaN in compound: convert to string sentinel since CatBoost can't take NaN in cat columns.
for c in categorical_features:
    df[c] = df[c].astype(str).fillna('NA')

print(df.shape, '/', len(features), 'features')

(540511, 15) / 14 features


## CV loop — same StratifiedGroupKFold(10) as the other models

In [3]:
cat_params = {
    'iterations': 5000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'task_type': 'GPU',
    'devices': '0',
    'random_seed': 123,
    'verbose': 0,
}

sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=123)
scores = []
best_iters = []
oof_preds = np.zeros(len(df))

X, y = df[features], df[target]
groups = (df['race'].astype(str) + '_' + df['year'].astype(str)).values
strat_y = (df['pitnextlap'].astype(str) + '_' + df['year'].astype(str))

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, strat_y, groups=groups)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=categorical_features,
        sample_weight=sample_weights[train_idx],
        early_stopping_rounds=50,
        verbose=0,
    )

    proba = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = proba
    score = roc_auc_score(y_val, proba)
    scores.append(score)
    best_iters.append(model.best_iteration_)
    print(f'Fold {fold}: AUC={score:.5f}  best_iter={model.best_iteration_}  sw_sum={sample_weights[train_idx].sum():.1f}')

print(f'\nMean AUC: {np.mean(scores):.5f} ± {np.std(scores):.5f}')
print(f'Mean best_iter: {int(np.mean(best_iters))}  Max best_iter: {int(np.max(best_iters))}')

oof_train = oof_preds[:train_df_length]
y_train_arr = y.iloc[:train_df_length].values
pd.DataFrame({'oof': oof_train, 'y': y_train_arr}).to_csv('./archive/catboost_oof.csv', index=False)
print(f'\nOOF AUC (train portion): {roc_auc_score(y_train_arr, oof_train):.5f}')
print('saved ./archive/catboost_oof.csv')

Default metric period is 5 because AUC is/are not implemented for GPU


Fold 0: AUC=0.91519  best_iter=864  sw_sum=513214.8


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1: AUC=0.91065  best_iter=363  sw_sum=507762.5


Default metric period is 5 because AUC is/are not implemented for GPU


...

Fold 7: AUC=0.92552  best_iter=433  sw_sum=511696.8


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 8: AUC=0.92706  best_iter=773  sw_sum=506126.5


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 9: AUC=0.91811  best_iter=677  sw_sum=504731.0

Mean AUC: 0.92650 ± 0.01005
Mean best_iter: 714  Max best_iter: 1086

OOF AUC (train portion): 0.93130
saved ./archive/catboost_oof.csv


## Correlation with XGB and LGBM OOFs

The whole point of CatBoost here is diversity. If the new OOF correlates >0.99 with the existing models we won't get blend uplift; if it's lower (say <0.97) we likely will.

In [4]:
xgb_oof  = pd.read_csv('./archive/xgb_oof.csv')['oof'].values
lgbm_oof = pd.read_csv('./archive/lgbm_oof.csv')['oof'].values
cat_oof  = oof_train

print(f'CatBoost OOF AUC:  {roc_auc_score(y_train_arr, cat_oof):.5f}')
print(f'XGB OOF AUC:       {roc_auc_score(y_train_arr, xgb_oof):.5f}')
print(f'LGBM OOF AUC:      {roc_auc_score(y_train_arr, lgbm_oof):.5f}')
print()
print(f'XGB vs LGBM corr:      {np.corrcoef(xgb_oof, lgbm_oof)[0,1]:.4f}')
print(f'XGB vs CatBoost corr:  {np.corrcoef(xgb_oof, cat_oof )[0,1]:.4f}')
print(f'LGBM vs CatBoost corr: {np.corrcoef(lgbm_oof, cat_oof)[0,1]:.4f}')

# Quick equal-weight 3-way blend preview
eq = (xgb_oof + lgbm_oof + cat_oof) / 3
print(f'\nEqual-weight 3-way blend OOF AUC: {roc_auc_score(y_train_arr, eq):.5f}')

CatBoost OOF AUC:  0.93130
XGB OOF AUC:       0.93514
LGBM OOF AUC:      0.93522

XGB vs LGBM corr:      0.9907
XGB vs CatBoost corr:  0.9571
LGBM vs CatBoost corr: 0.9650

Equal-weight 3-way blend OOF AUC: 0.93634


## Final model + test predictions

In [5]:
final_params = {**cat_params}
final_params['iterations'] = int(np.max(best_iters) * 1.10)

cat_final_model = CatBoostClassifier(**final_params)
cat_final_model.fit(X, y, cat_features=categorical_features, sample_weight=sample_weights, verbose=0)
print('trained final model with iterations =', final_params['iterations'])

pd.Series(cat_final_model.feature_importances_, index=features).sort_values(ascending=False)

Default metric period is 5 because AUC is/are not implemented for GPU


trained final model with iterations = 1194


year_cat                  35.413969
stint                     14.802006
tyrelife                  11.045089
year                       6.634517
laptime_delta              5.746141
compound                   4.555787
raceprogress               4.380146
race                       4.269676
lapnumber                  3.091348
laptime_(s)                2.819607
cumulative_degradation     2.292488
position                   1.877384
position_change            1.696793
pitstop                    1.375047
dtype: float64

In [6]:
test = (test_df
            .pipe(copy_data)
            .pipe(clean_data)
            .pipe(remove_duplicates)
            .pipe(make_new_features))
for c in categorical_features:
    test[c] = test[c].astype(str).fillna('NA')

preds = cat_final_model.predict_proba(test[features])[:, 1]
pd.DataFrame({'id': test_df['id'].values, 'pred': preds}).to_csv('./archive/catboost_test.csv', index=False)
print('saved ./archive/catboost_test.csv')

saved ./archive/catboost_test.csv
